%md
# Exploração: Rede e Atendimento
Perfilamento de `ocorrencias_rede` e `atendimentos` (Bronze) para definir as regras da `silver_rede_atendimento`.

In [0]:
%run ./funcoes_comuns

In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOGO}.bronze").show(truncate=False)

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOGO}.silver").show(truncate=False)

## Ocorrências

## Chave, valores vazios e espaços
**O que avaliamos:** se a chave `id_ocorrencia` é única e sem nulos, se há linhas duplicadas e, em cada coluna, quantos valores são nulos, vêm escritos como "sem valor" (`-`, `N/A`, `NULL`...), têm espaços nas pontas e quantos valores distintos existem.

In [0]:
oc = ler_bronze(CATALOGO, "ocorrencias_rede")
N_OC = oc.count()
print("linhas:", N_OC)

# Chave primária: única e sem nulos?
print("id_ocorrencia distintos:", oc.select("id_ocorrencia").distinct().count())
print("id_ocorrencia nulos:", oc.filter(F.col("id_ocorrencia").isNull()).count())

# Linhas duplicadas ignorando o id e as colunas de rastreio
COLS = [c for c in oc.columns if not c.startswith("_")]
COLS_SEM_ID = [c for c in COLS if c != "id_ocorrencia"]
print("duplicadas (sem id):", N_OC - oc.select(COLS_SEM_ID).distinct().count())

# Por coluna, numa passada só: nulos, "sem valor" escrito, espaço nas pontas, distintos
SEM_VALOR = ["", "-", "--", "N/A", "NA", "NULL", "NONE", "NAN", "-1"]
exprs = []
for c in COLS:
    col = F.col(c)
    exprs += [
        F.sum(F.when(col.isNull(), 1).otherwise(0)).alias(f"{c}__nulos"),
        F.sum(F.when(F.upper(F.trim(col)).isin(SEM_VALOR), 1).otherwise(0)).alias(f"{c}__sem_valor"),
        F.sum(F.when(col.rlike(r"^[\s\u00A0]|[\s\u00A0]$"), 1).otherwise(0)).alias(f"{c}__espacos"),
        F.countDistinct(col).alias(f"{c}__distintos"),
    ]
r = oc.agg(*exprs).first().asDict()
resumo = [(c, r[f"{c}__nulos"], r[f"{c}__sem_valor"], r[f"{c}__espacos"], r[f"{c}__distintos"]) for c in COLS]
display(spark.createDataFrame(resumo, ["coluna", "nulos", "sem_valor_escrito", "espaco_pontas", "distintos"]))

**O que descobrimos:**
- 180.000 linhas, `id_ocorrencia` único e sem nulos.
- Nenhuma linha duplicada (mesmo ignorando o id).
- Só a coluna `causa` tem valores ausentes, **5.400** no total: 1.356 já chegam como nulo na Bronze (eram vazios no CSV). 4.044 vêm escritos como "sem valor": `-`, `N/A` e `NULL`.
- Nenhuma coluna tem espaços nas pontas.
- 41.512 UCs distintas e 122 alimentadores (o mesmo número do cadastro).

## Domínios e formatos
**O que avaliamos:** quais valores e com que frequência aparecem nas colunas de categoria (`tipo` e `causa`) e se os IDs, as datas e a duração seguem sempre o mesmo formato. Para comparar formatos, trocamos cada dígito por `9` (ou cada número por `N`, na duração): um formato diferente aparece como uma linha a mais no resultado.

In [0]:
# Valores das colunas de domínio
for c in ["tipo", "causa"]:
    print(f"\n== {c}")
    oc.groupBy(c).count().orderBy(F.desc("count")).show(50, truncate=False)

# Formato dos campos (cada dígito vira 9)
for c in ["id_ocorrencia", "id_uc", "alimentador", "data_abertura", "data_encerramento"]:
    print(f"\n== formato de {c}")
    (oc.groupBy(F.regexp_replace(c, r"\d", "9").alias("formato"))
       .count().orderBy(F.desc("count")).show(10, truncate=False))

# Duração: cada número vira N, para ver o texto ao redor
print("\n== formato de duracao_min")
(oc.groupBy(F.regexp_replace("duracao_min", r"\d+", "N").alias("formato"))
   .count().orderBy(F.desc("count")).show(10, truncate=False))

**O que descobrimos:**
- `tipo`: 5 valores limpos, sem variações de escrita. "FALTA DE ENERGIA" domina (68%).
- `causa`: 6 causas reais + **5.400 sem valor**: 1.356 nulos reais, 1.363 `-`, 1.341 `N/A` e 1.340 escritos como "NULL" (o `show()` exibe nulos reais e o texto "NULL" do mesmo jeito).
  - `NAO IDENTIFICADA` já é uma causa real (houve investigação, sem conclusão). Os vazios virarão `NAO INFORMADA`, que significa "campo não preenchido".
  - "SOBRECARGA" aparece em `tipo` e em `causa` com sentidos diferentes (efeito × origem). Não é erro.
- IDs (`OC99999999`, `UC999999`) e as duas datas (`aaaa-mm-dd hh:mm:ss`) têm formato único.
- `alimentador` segue sempre `ALM-<município>-999`.
- `duracao_min` tem dois formatos: só o número (163.800) e número + " min" (**16.200**). É preciso remover o " min" antes de converter para inteiro.

## Conversões e coerência de tempo
**O que avaliamos:** se a duração e as datas convertem sem perder valores, se a duração informada bate com o tempo entre abertura e encerramento e se algum encerramento vem antes da abertura. Também olhamos os casos extremos (abaixo de 3 min, acima de 24h e exatamente 48h) e procuramos interrupções sobrepostas na mesma UC, quando uma começa antes da anterior terminar, o que contaria tempo em dobro no DEC. 

Usamos `try_cast` para que um valor sujo vire nulo em vez de quebrar a célula.

In [0]:
oc_t = (oc
    .withColumn("dur", F.expr("try_cast(regexp_replace(duracao_min, ' min$', '') as int)"))
    .withColumn("abertura", F.expr("try_cast(data_abertura as timestamp)"))
    .withColumn("encerramento", F.expr("try_cast(data_encerramento as timestamp)"))
    .withColumn("dur_calc", (F.unix_timestamp("encerramento") - F.unix_timestamp("abertura")) / 60))

def contar(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

# Conversões, coerência e extremos
r = oc_t.agg(
    contar(F.col("dur").isNull()).alias("duracao_nao_converte"),
    contar(F.col("abertura").isNull()).alias("abertura_nao_converte"),
    contar(F.col("encerramento").isNull()).alias("encerramento_nao_converte"),
    contar(F.col("encerramento") <= F.col("abertura")).alias("encerramento_antes_abertura"),
    contar(F.col("dur") != F.col("dur_calc")).alias("duracao_diverge_datas"),
    contar(F.col("dur") < 3).alias("abaixo_3min"),
    contar(F.col("dur") > 1440).alias("acima_24h"),
    contar(F.col("dur") == 2880).alias("exatamente_48h"),
    F.min("dur").alias("duracao_min"), F.max("dur").alias("duracao_max"),
    F.min("abertura").alias("primeira_abertura"), F.max("abertura").alias("ultima_abertura"),
).first()
for k, v in r.asDict().items():
    print(f"{k:<30}{v}")

# Acima de 24h por causa
print("\n== acima de 24h por causa")
oc_t.filter(F.col("dur") > 1440).groupBy("causa").count().orderBy(F.desc("count")).show(truncate=False)

# Sobreposição: na mesma UC, a interrupção começa antes da anterior terminar?
w = Window.partitionBy("id_uc").orderBy("abertura", "id_ocorrencia")
sob = (oc_t.withColumn("fim_anterior", F.lag("encerramento").over(w))
           .filter(F.col("abertura") < F.col("fim_anterior")))
r2 = sob.agg(
    F.count("*").alias("sobrepostas"),
    contar(F.col("encerramento") <= F.col("fim_anterior")).alias("contidas_inteiras"),
    F.sum((F.unix_timestamp(F.least("encerramento", "fim_anterior")) - F.unix_timestamp("abertura")) / 60).alias("minutos_em_dobro"),
).first()
print("\n== sobreposições na mesma UC")
for k, v in r2.asDict().items():
    print(f"{k:<30}{v}")

**O que descobrimos:**
- Duração e datas convertem 100% depois de remover o " min". Nenhum encerramento vem antes da abertura.
- A duração informada é exatamente o tempo entre abertura e encerramento em todas as linhas, então é confiável.
- O período vai de 01/07/2022 a 30/06/2026, como esperado.
- Nenhuma interrupção abaixo de 3 min. A menor dura 5 min.
- 274 interrupções passam de 24h, 230 delas por causa climática. Serão marcadas com `flag_longa_duracao`. O Padrão de Limpeza diz 244, mas o número correto é 274.
- 30 interrupções duram exatamente 48h, que é também a maior duração da tabela. Isso indica um teto no sistema de origem, e a duração real pode ter sido maior. Serão marcadas com `flag_duracao_teto`.
- 329 interrupções começam antes da anterior terminar na mesma UC, e 153 delas ficam inteiras dentro da anterior. Somadas, contam 43.144 minutos em dobro. Serão marcadas com `flag_sobreposicao`, sem apagar nada. A decisão de juntar os intervalos fica para a Gold.

## Integridade com o cadastro
**O que avaliamos:** se toda ocorrência aponta para uma UC que existe na Silver do cadastro (as que não existem são chamadas de órfãs) e se o alimentador informado na ocorrência é o mesmo alimentador da UC no cadastro.

In [0]:
uc = (spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")
           .select("id_uc", F.col("alimentador").alias("alimentador_cad")))
print("UCs no cadastro:", uc.count())

# Órfãs: ocorrências cuja UC não existe no cadastro limpo
orfas = oc_t.join(uc, "id_uc", "left_anti")
print("ocorrencias orfas:", orfas.count())
orfas.select("id_ocorrencia", "id_uc").show(5, truncate=False)

# Alimentador da ocorrência diferente do alimentador da UC no cadastro
dif_alim = oc_t.join(uc, "id_uc").filter(F.col("alimentador") != F.col("alimentador_cad"))
print("alimentador diferente do cadastro:", dif_alim.count())
dif_alim.select("id_ocorrencia", "id_uc", "alimentador", "alimentador_cad").show(5, truncate=False)

# Alimentadores que existem nas ocorrências mas não no cadastro
alim_cad = uc.select(F.col("alimentador_cad").alias("alimentador")).distinct()
print("alimentadores fora do cadastro:", oc_t.select("alimentador").distinct().join(alim_cad, "alimentador", "left_anti").count())

**O que descobrimos:**
- Todas as ocorrências apontam para UCs que existem na Silver do cadastro (45.000 UCs). Nenhuma ocorrência órfã.
- O alimentador da ocorrência é sempre o mesmo alimentador da UC no cadastro, e não há alimentador fora do cadastro.
- Na Silver, o `separar_orfaos` roda mesmo assim, como garantia, e a quarentena fica vazia.

## Distribuição no tempo
**O que avaliamos:** quantas ocorrências acontecem por mês, para encontrar períodos fora do normal. Junto, contamos por mês as causas climáticas, as interrupções acima de 24h e as sobreposições, para ver se esses casos se concentram em algum período. No mês com mais ocorrências, olhamos quais municípios foram mais afetados.

In [0]:
# Volume por mês, com climáticas, acima de 24h e sobreposições
mensal = (oc_t.withColumn("mes", F.date_format("abertura", "yyyy-MM"))
    .join(sob.select("id_ocorrencia").withColumn("sobreposta", F.lit(1)), "id_ocorrencia", "left")
    .groupBy("mes")
    .agg(F.count("*").alias("ocorrencias"),
         contar(F.col("causa") == "CLIMATICA").alias("climaticas"),
         contar(F.col("dur") > 1440).alias("acima_24h"),
         contar(F.col("sobreposta") == 1).alias("sobrepostas"))
    .orderBy("mes"))
mensal.show(48, truncate=False)

n_meses = mensal.count()
print("meses:", n_meses, "| media de ocorrencias por mes:", round(N_OC / n_meses))

# Mês com mais ocorrências: municípios mais afetados (código do município no alimentador)
mes_pico = mensal.orderBy(F.desc("ocorrencias")).first()["mes"]
print(f"\n== mes com mais ocorrencias: {mes_pico}")
(oc_t.filter(F.date_format("abertura", "yyyy-MM") == mes_pico)
     .groupBy(F.split("alimentador", "-")[1].alias("municipio")).count()
     .orderBy(F.desc("count")).show(10, truncate=False))

# Alimentadores no mês de pico e fora dele
oc_alim = oc_t.withColumn("no_pico", F.date_format("abertura", "yyyy-MM") == mes_pico)
print(f"\n== alimentadores com mais ocorrencias em {mes_pico}")
(oc_alim.groupBy("alimentador")
        .agg(F.count("*").alias("total_4_anos"),
             contar(F.col("no_pico")).alias("no_pico"),
             contar(~F.col("no_pico")).alias("fora_do_pico"))
        .orderBy(F.desc("no_pico")).show(5, truncate=False))

top3 = [r["alimentador"] for r in oc_alim.filter("no_pico").groupBy("alimentador").count()
        .orderBy(F.desc("count")).limit(3).collect()]
n_pico = oc_alim.filter("no_pico").count()
n_top3 = oc_alim.filter(F.col("no_pico") & F.col("alimentador").isin(top3)).count()
print(f"ocorrencias no pico: {n_pico} | nos 3 alimentadores {top3}: {n_top3} ({round(n_top3 / n_pico * 100)}%)")

print("\n== alimentadores com mais ocorrencias fora do pico")
oc_alim.filter(~F.col("no_pico")).groupBy("alimentador").count().orderBy(F.desc("count")).show(3, truncate=False)

**O que descobrimos:**
**O que descobrimos:**
- Um mês normal tem cerca de 3.370 ocorrências, das quais cerca de 550 são climáticas. A média geral de 3.750 está inflada pelos meses anômalos.
- Julho de 2025 é um evento extremo: 17.844 ocorrências, mais de 5 vezes o normal, sendo 17.352 climáticas. O mês concentra 225 das 274 interrupções acima de 24h e 266 das 329 sobreposições. Fora dele, esses casos aparecem 1 ou 2 vezes por mês.
- O evento foi localizado: 12.836 das ocorrências do mês (72%) estão em só três alimentadores, ALM-REC-008 (6.887), ALM-SAO-004 (3.351) e ALM-JAB-004 (2.598). Um alimentador comum teve cerca de 60 no mesmo mês. Fora de julho, os três são alimentadores normais, com cerca de 1.320 ocorrências cada em 4 anos.
- Por isso Recife, São Lourenço e Jaboatão lideram o mês. São Lourenço foi o mais atingido em proporção: cerca de 40% de tudo que o município teve em 4 anos aconteceu nesse mês.
- Novembro de 2025 e fevereiro de 2026 também fogem do normal, com cerca de 5.300 ocorrências cada. Mas as climáticas subiram só cerca de 300 nesses meses, então o excesso tem outra origem, investigada mais adiante.
- Não é problema de limpeza: as linhas ficam como estão na Silver. É informação importante para o DEC/FEC na Gold.

## Coerência entre colunas
**O que avaliamos:** se as combinações de tipo e causa fazem sentido, quantas ocorrências pertencem a eventos coletivos (várias UCs do mesmo alimentador interrompidas no mesmo horário, que são um único evento na rede) e se alguma UC tem um número de interrupções fora do comum.

In [0]:
# Tipo x causa
print("== tipo x causa")
(oc_t.withColumn("causa_v", F.coalesce("causa", F.lit("(nulo)")))
     .groupBy("tipo").pivot("causa_v").count().na.fill(0).orderBy("tipo").show(truncate=False))

# Eventos coletivos: mesmo alimentador e mesma abertura
ev_col = oc_t.groupBy("alimentador", "abertura").agg(F.count("*").alias("ucs_afetadas"))
r = ev_col.agg(
    F.count("*").alias("eventos_distintos"),
    contar(F.col("ucs_afetadas") > 1).alias("eventos_com_mais_de_1_uc"),
    F.sum(F.when(F.col("ucs_afetadas") > 1, F.col("ucs_afetadas")).otherwise(0)).alias("linhas_em_eventos_coletivos"),
    F.max("ucs_afetadas").alias("maior_evento_ucs"),
).first()
print("\n== eventos coletivos")
for k, v in r.asDict().items():
    print(f"{k:<30}{v}")
ev_col.orderBy(F.desc("ucs_afetadas")).show(5, truncate=False)

# Interrupções por UC no período e no mês
por_uc = oc_t.groupBy("id_uc").agg(F.count("*").alias("n"))
r = por_uc.agg(
    F.min("n").alias("min_por_uc"),
    F.expr("percentile_approx(n, 0.5)").alias("mediana_por_uc"),
    F.expr("percentile_approx(n, 0.99)").alias("p99_por_uc"),
    F.max("n").alias("max_por_uc"),
).first()
print("\n== interrupcoes por UC (4 anos)")
for k, v in r.asDict().items():
    print(f"{k:<30}{v}")
print("UCs sem nenhuma ocorrencia:", uc.count() - por_uc.count())

por_uc_mes = oc_t.groupBy("id_uc", F.date_format("abertura", "yyyy-MM").alias("mes")).count()
print("\n== UCs com mais interrupcoes num unico mes")
por_uc_mes.orderBy(F.desc("count")).show(5, truncate=False)

**O que descobrimos:**
- Todas as combinações de tipo e causa aparecem, e as causas se distribuem quase por igual entre os tipos. A única exceção é CLIMATICA, que pesa mais em FALTA DE ENERGIA. Não há combinação impossível.
- Eventos coletivos quase não existem: as 180.000 linhas formam 179.265 eventos distintos (mesmo alimentador e mesmo horário). Só 712 eventos atingem mais de uma UC, e o maior atinge 3. Na prática, cada linha é um evento isolado.
- Entre as UCs que tiveram alguma interrupção, metade teve até 3 em 4 anos (mediana) e 99% tiveram até 19. A que mais teve chegou a 81. Valores altos são plausíveis para UCs em trechos de rede ruins e ficam como estão.
- 3.488 UCs (7,8% do cadastro) não tiveram nenhuma interrupção no período.
- No pior mês de uma única UC, foram 12 interrupções, todas em julho de 2025.

## Novembro de 2025 e fevereiro de 2026
**O que avaliamos:** esses dois meses têm cerca de 2.000 ocorrências a mais que o normal, mas o clima explica só uma parte. Comparamos com os meses vizinhos por tipo, causa, município, dia do mês e duração para descobrir de onde vem o excesso. Uma hipótese é a troca de medidores feita entre setembro de 2025 e março de 2026.

In [0]:
MESES = ["2025-10", "2025-11", "2025-12", "2026-01", "2026-02", "2026-03"]
oc_m = (oc_t.withColumn("mes", F.date_format("abertura", "yyyy-MM"))
            .withColumn("causa_v", F.coalesce("causa", F.lit("(nulo)")))
            .withColumn("municipio", F.split("alimentador", "-")[1])
            .filter(F.col("mes").isin(MESES)))

for c in ["tipo", "causa_v", "municipio"]:
    print(f"\n== {c} por mes")
    (oc_m.groupBy(c).pivot("mes", MESES).count().na.fill(0)
         .orderBy(F.desc("2025-11")).show(15, truncate=False))

for m in ["2025-11", "2026-02"]:
    print(f"\n== {m}: dias com mais ocorrencias")
    (oc_m.filter(F.col("mes") == m).groupBy(F.dayofmonth("abertura").alias("dia")).count()
         .orderBy(F.desc("count")).show(5, truncate=False))

print("\n== duracao por mes")
(oc_m.groupBy("mes").agg(
        F.count("*").alias("ocorrencias"),
        F.round(F.avg("dur"), 1).alias("duracao_media"),
        F.expr("percentile_approx(dur, 0.5)").alias("duracao_mediana"))
     .orderBy("mes").show(truncate=False))

**O que descobrimos:**
- O excesso de cerca de 2.000 ocorrências é uniforme: todos os tipos, todas as causas (inclusive as sem valor) e todos os municípios crescem na mesma proporção, cerca de 55% acima dos meses vizinhos.
- Não há concentração em nenhum dia do mês, e a duração média e a mediana ficam iguais às dos meses normais.
- Os atendimentos de FALTA DE ENERGIA não sobem nesses meses (cerca de 2.000, como sempre, ver a seção de tempo dos atendimentos). Ou seja, há mais interrupções registradas, mas os clientes não ligaram mais.
- Um crescimento igual em tudo, sem reflexo nas reclamações, não se parece com um evento real na rede. A hipótese da troca de medidores é testada no bloco seguinte.
- Não é problema de limpeza: as linhas ficam como estão na Silver, mas o ponto precisa ser considerado no cálculo do DEC/FEC.

## Troca de medidores e os meses de novembro e fevereiro
**O que avaliamos:** se o excesso de ocorrências em novembro de 2025 e fevereiro de 2026 coincide com instalações de medidores, que exigem desligar a UC. Contamos as instalações por mês e quantas ocorrências aconteceram na mesma UC e no mesmo dia de uma instalação.

In [0]:
med = spark.table(f"{CATALOGO}.silver.tb_medidores")
print("colunas:", med.columns)

# Instalações de medidores por mês
print("\n== instalacoes por mes")
(med.groupBy(F.date_format("data_instalacao", "yyyy-MM").alias("mes")).count()
    .filter(F.col("mes").between("2025-06", "2026-06")).orderBy("mes").show(truncate=False))

# Ocorrências na mesma UC e no mesmo dia de uma instalação
inst = med.select("id_uc", F.col("data_instalacao").alias("dia")).distinct().withColumn("instalacao_no_dia", F.lit(1))
(oc_t.withColumn("dia", F.to_date("abertura"))
     .join(inst, ["id_uc", "dia"], "left")
     .groupBy(F.date_format("abertura", "yyyy-MM").alias("mes"))
     .agg(F.count("*").alias("ocorrencias"),
          contar(F.col("instalacao_no_dia") == 1).alias("no_dia_de_instalacao"))
     .filter(F.col("mes").between("2025-06", "2026-06")).orderBy("mes").show(truncate=False))

**O que descobrimos:**
- A troca de medidores aconteceu de setembro de 2025 a março de 2026, com cerca de 900 instalações por mês, de forma constante. Antes disso, eram cerca de 5 a 10 por mês.
- Novembro de 2025 e fevereiro de 2026 não têm mais instalações que os outros meses da campanha.
- Só 6 ocorrências em novembro e 4 em fevereiro aconteceram na mesma UC e no mesmo dia de uma instalação.
- A troca de medidores não explica o excesso. A origem das cerca de 2.000 ocorrências a mais nesses dois meses continua sem explicação nos dados disponíveis, e o ponto fica registrado para a Gold.

## Atendimentos

## Chave, valores vazios e espaços
**O que avaliamos:** como a tabela foi gravada na Bronze (principalmente o histórico, que pode estar como texto ou como lista), se o `protocolo` é único e sem nulos, se há atendimentos duplicados e, em cada coluna, quantos valores são nulos, vêm escritos como "sem valor", têm espaços nas pontas e quantos valores distintos existem.

In [0]:
at = ler_bronze(CATALOGO, "atendimentos")
N_AT = at.count()
print("linhas:", N_AT)
print("tipos das colunas:")
for c, t in at.dtypes:
    print(f"  {c:<25}{t}")

# Chave primária
print("\nprotocolo distintos:", at.select("protocolo").distinct().count())
print("protocolo nulos:", at.filter(F.col("protocolo").isNull()).count())

# Duplicados ignorando o protocolo, o histórico e o rastreio
COLS_AT = [c for c in at.columns if not c.startswith("_") and c != "historico"]
COLS_AT_SEM_ID = [c for c in COLS_AT if c != "protocolo"]
print("duplicados (sem protocolo):", N_AT - at.select(COLS_AT_SEM_ID).distinct().count())

# Histórico vazio ou nulo
tipo_hist = dict(at.dtypes)["historico"]
tamanho = F.length("historico") if tipo_hist.startswith("string") else F.size("historico")
print("historico nulo:", at.filter(F.col("historico").isNull()).count())
print("historico vazio:", at.filter(tamanho <= 2 if tipo_hist.startswith("string") else tamanho == 0).count())

# Por coluna: nulos, "sem valor" escrito, espaços, distintos
exprs = []
for c in COLS_AT:
    col = F.col(c).cast("string")
    exprs += [
        contar(col.isNull()).alias(f"{c}__nulos"),
        contar(F.upper(F.trim(col)).isin(SEM_VALOR)).alias(f"{c}__sem_valor"),
        contar(col.rlike(r"^[\s\u00A0]|[\s\u00A0]$")).alias(f"{c}__espacos"),
        F.countDistinct(col).alias(f"{c}__distintos"),
    ]
r = at.agg(*exprs).first().asDict()
print(f"\n{'coluna':<20}{'nulos':>8}{'sem_valor':>11}{'espacos':>9}{'distintos':>11}")
for c in COLS_AT:
    print(f"{c:<20}{r[c+'__nulos']:>8}{r[c+'__sem_valor']:>11}{r[c+'__espacos']:>9}{r[c+'__distintos']:>11}")

**O que descobrimos:**
- 318.871 atendimentos, `protocolo` único e sem nulos, nenhum atendimento duplicado.
- Na Bronze, todas as colunas estão como texto e o `historico` já é uma lista de eventos, com os campos em texto. Não é preciso converter o histórico de JSON.
- Nenhum histórico vazio ou nulo.
- A única coluna com nulos é `nota_satisfacao` (9.566). Nenhuma coluna tem "sem valor" escrito ou espaços nas pontas.
- 44.940 UCs distintas fizeram atendimentos, quase todo o cadastro.

## Domínios e formatos
**O que avaliamos:** quais valores aparecem em canal, assunto e subassunto, se cada subassunto pertence sempre ao mesmo assunto e se ids e datas seguem um único formato. Na nota de satisfação, conferimos como ela está escrita, se converte para número, se fica entre 1 e 5 e como os nulos se distribuem.

In [0]:
for c in ["canal", "assunto"]:
    print(f"\n== {c}")
    at.groupBy(c).count().orderBy(F.desc("count")).show(30, truncate=False)

print("\n== assunto x subassunto")
at.groupBy("assunto", "subassunto").count().orderBy("assunto", F.desc("count")).show(60, truncate=False)
print("subassuntos em mais de um assunto:",
      at.groupBy("subassunto").agg(F.countDistinct("assunto").alias("n")).filter("n > 1").count())

for c in ["protocolo", "id_uc", "data_abertura", "data_fechamento"]:
    print(f"\n== formato de {c}")
    (at.groupBy(F.regexp_replace(F.col(c).cast("string"), r"\d", "9").alias("formato"))
       .count().orderBy(F.desc("count")).show(10, truncate=False))

# Nota de satisfação
nota_s = F.col("nota_satisfacao").cast("string")
print("\n== formato da nota")
at.groupBy(F.regexp_replace(nota_s, r"\d", "9").alias("formato")).count().orderBy(F.desc("count")).show(10, truncate=False)

at_n = at.withColumn("nota", F.expr("try_cast(cast(nota_satisfacao as string) as double)"))
r = at_n.agg(
    contar(F.col("nota_satisfacao").isNull()).alias("nulas"),
    contar(F.col("nota_satisfacao").isNotNull() & F.col("nota").isNull()).alias("nao_converte"),
    contar((F.col("nota") < 1) | (F.col("nota") > 5)).alias("fora_1_a_5"),
    contar(F.round("nota", 1) != F.col("nota")).alias("mais_de_1_casa"),
    F.min("nota").alias("min"), F.max("nota").alias("max"),
    F.round(F.avg("nota"), 2).alias("media"),
).first()
print("\n== nota")
for k, v in r.asDict().items():
    print(f"{k:<20}{v}")

print("\n== notas nulas por canal")
(at.groupBy("canal").agg(F.count("*").alias("total"), contar(F.col("nota_satisfacao").isNull()).alias("nulas"))
   .withColumn("pct_nulas", F.round(F.col("nulas") / F.col("total") * 100, 1))
   .orderBy("canal").show(truncate=False))

**O que descobrimos:**
- `canal`: 5 valores limpos, com volumes quase iguais (cerca de 64 mil cada).
- `assunto`: 6 valores limpos. FALTA DE ENERGIA é o maior (101 mil), os outros têm cerca de 43 mil cada.
- `subassunto`: 15 valores em 17 combinações com o assunto. VISTORIA aparece em três assuntos (DANO ELETRICO, GERACAO DISTRIBUIDA e LIGACAO NOVA), o que é legítimo. Por isso o subassunto só deve ser analisado junto com o assunto.
- Protocolo (`AT` + 9 dígitos), UC e as duas datas têm formato único.
- A nota tem sempre uma casa decimal, converte 100% para número e fica entre 1,3 e 5,0, com média 4,04. Na Silver, vira decimal com uma casa.
- As 9.566 notas nulas (3%) se distribuem por igual entre os canais, então não dependem do canal de atendimento. Ficam nulas, nunca zero.

## Tempo e integridade com o cadastro
**O que avaliamos:** se as datas convertem, se algum fechamento vem antes da abertura, quanto tempo os atendimentos levam e se há datas fora do período da base. Depois conferimos se todo atendimento aponta para uma UC do cadastro e como o volume se distribui por mês e assunto, para ver se os picos das ocorrências aparecem também aqui.

In [0]:
at_t = (at.withColumn("abertura", F.expr("try_cast(data_abertura as timestamp)"))
          .withColumn("fechamento", F.expr("try_cast(data_fechamento as timestamp)"))
          .withColumn("horas", (F.unix_timestamp("fechamento") - F.unix_timestamp("abertura")) / 3600))

r = at_t.agg(
    contar(F.col("abertura").isNull()).alias("abertura_nao_converte"),
    contar(F.col("fechamento").isNull()).alias("fechamento_nao_converte"),
    contar(F.col("fechamento") < F.col("abertura")).alias("fechamento_antes_abertura"),
    contar(F.col("fechamento") == F.col("abertura")).alias("fechamento_igual_abertura"),
    F.round(F.min("horas"), 2).alias("horas_min"),
    F.expr("percentile_approx(horas, 0.5)").alias("horas_mediana"),
    F.expr("percentile_approx(horas, 0.99)").alias("horas_p99"),
    F.round(F.max("horas"), 1).alias("horas_max"),
    F.min("abertura").alias("primeira_abertura"), F.max("abertura").alias("ultima_abertura"),
    contar(F.col("fechamento") > F.lit("2026-06-30 23:59:59").cast("timestamp")).alias("fecha_apos_jun26"),
).first()
for k, v in r.asDict().items():
    print(f"{k:<28}{v}")

# Órfãos
orfaos_at = at_t.join(uc.select("id_uc"), "id_uc", "left_anti")
print("\natendimentos orfaos:", orfaos_at.count())
orfaos_at.select("protocolo", "id_uc").show(5, truncate=False)

# Volume por mês e assunto
print("\n== atendimentos por mes e assunto")
assuntos = [r["assunto"] for r in at.select("assunto").distinct().orderBy("assunto").collect()]
(at_t.withColumn("mes", F.date_format("abertura", "yyyy-MM"))
     .groupBy("mes").pivot("assunto", assuntos).count().na.fill(0)
     .withColumn("total", sum(F.col(f"`{a}`") for a in assuntos))
     .orderBy("mes").show(48, truncate=False))

**O que descobrimos:**
- As datas convertem 100%, e nenhum fechamento vem antes ou no mesmo instante da abertura.
- Os atendimentos duram de 1 a 719 horas. A mediana é de 125 horas (cerca de 5 dias), e 99% fecham em até 591 horas. Nenhum passa de 30 dias, o que sugere um prazo máximo no processo.
- As aberturas vão de 01/07/2022 a 30/06/2026. 1.073 atendimentos fecham depois de junho de 2026: foram abertos no fim do período e fechados depois, o que é legítimo.
- Nenhum atendimento órfão: todos apontam para UCs do cadastro.
- Um mês normal tem cerca de 6.400 atendimentos, sendo cerca de 2.000 de FALTA DE ENERGIA.
- Julho de 2025 tem 18.061 atendimentos, quase 3 vezes o normal. Mas todos os assuntos crescem igualmente, inclusive FATURA e LIGACAO NOVA, que não têm relação com um evento climático. É um comportamento estranho, que fica registrado para a análise na Gold.
- Novembro de 2025 e fevereiro de 2026 são meses normais nos atendimentos, apesar do excesso nas ocorrências.

## Atendimentos de geração distribuída
**O que avaliamos:** se os atendimentos sobre geração distribuída vêm de UCs que de fato têm GD. Comparamos, para cada assunto, a parcela de atendimentos feitos por UCs com GD com a parcela dessas UCs no cadastro.

In [0]:
gd = ler_bronze(CATALOGO, "geracao_distribuida").select("id_uc").distinct().withColumn("tem_gd", F.lit(True))
n_gd = gd.count()
print("UCs com GD:", n_gd, "| parcela do cadastro (%):", round(n_gd / uc.count() * 100, 1))

at_gd = at.join(gd, "id_uc", "left").withColumn("tem_gd", F.coalesce("tem_gd", F.lit(False)))

print("\n== atendimentos de UCs com GD, por assunto")
(at_gd.groupBy("assunto")
      .agg(F.count("*").alias("atendimentos"), contar(F.col("tem_gd")).alias("de_uc_com_gd"))
      .withColumn("pct_uc_com_gd", F.round(F.col("de_uc_com_gd") / F.col("atendimentos") * 100, 1))
      .orderBy("assunto").show(truncate=False))

print("\n== assunto GERACAO DISTRIBUIDA, por subassunto")
(at_gd.filter(F.col("assunto") == "GERACAO DISTRIBUIDA").groupBy("subassunto")
      .agg(F.count("*").alias("atendimentos"), contar(F.col("tem_gd")).alias("de_uc_com_gd"))
      .withColumn("pct_uc_com_gd", F.round(F.col("de_uc_com_gd") / F.col("atendimentos") * 100, 1))
      .orderBy("subassunto").show(truncate=False))

**O que descobrimos:**
- 3.510 UCs têm GD, 7,8% do cadastro.
- Em todos os assuntos, a parcela de atendimentos feitos por UCs com GD fica entre 7,6% e 8,0%, ou seja, igual à parcela delas no cadastro. Isso vale também para o próprio assunto GERACAO DISTRIBUIDA (7,8%) e para cada subassunto dele, inclusive CREDITOS, que só faria sentido para quem tem GD.
- Na prática, 92% dos atendimentos sobre GD vêm de UCs sem GD cadastrada. O assunto do atendimento não tem relação com o perfil da UC.
- Os atendimentos ficam como estão. Mas o assunto do atendimento não deve ser usado como sinal de característica da UC (por exemplo, para identificar quem tem GD) nas análises da Gold.

## Histórico dos atendimentos
**O que avaliamos:** abrimos a lista de eventos de cada atendimento, uma linha por evento, como ficará na tabela filha. Conferimos quantos eventos existem, os formatos e valores de cada campo e a coerência com o atendimento: sequência contínua começando em 1, sem eventos repetidos, datas em ordem e dentro do período do atendimento, último evento na data de fechamento e canal igual ao do atendimento. Por fim, olhamos se a ordem das ações faz sentido (abertura primeiro, encerramento por último).

In [0]:
# Lista de eventos: converte de texto se necessário
if tipo_hist.startswith("string"):
    SCHEMA_HIST = "array<struct<sequencia:string,data:string,acao:string,canal:string>>"
    at_h = at.withColumn("hist", F.from_json("historico", SCHEMA_HIST))
    print("historico que nao virou lista:", at_h.filter(F.col("historico").isNotNull() & F.col("hist").isNull()).count())
else:
    at_h = at.withColumn("hist", F.col("historico"))

print("\n== eventos por atendimento")
at_h.groupBy(F.size("hist").alias("n_eventos")).count().orderBy("n_eventos").show(truncate=False)

ev = (at_h.select("protocolo", "data_abertura", "data_fechamento", F.col("canal").alias("canal_at"),
                  F.posexplode("hist").alias("pos", "e"))
          .select("protocolo", "data_abertura", "data_fechamento", "canal_at", "pos",
                  F.col("e.sequencia").cast("string").alias("sequencia"),
                  F.col("e.data").cast("string").alias("data"),
                  F.col("e.acao").alias("acao"), F.col("e.canal").alias("canal")))
N_EV = ev.count()
print("eventos:", N_EV)

for c in ["sequencia", "data"]:
    print(f"\n== formato de {c}")
    ev.groupBy(F.regexp_replace(c, r"\d", "9").alias("formato")).count().orderBy(F.desc("count")).show(10, truncate=False)
for c in ["acao", "canal"]:
    print(f"\n== {c}")
    ev.groupBy(c).count().orderBy(F.desc("count")).show(20, truncate=False)

# Coerência com o atendimento
ev_t = (ev.withColumn("seq", F.expr("try_cast(sequencia as int)"))
          .withColumn("data_ts", F.expr("try_cast(data as timestamp)"))
          .withColumn("abertura", F.expr("try_cast(data_abertura as timestamp)"))
          .withColumn("fechamento", F.expr("try_cast(data_fechamento as timestamp)")))
w = Window.partitionBy("protocolo").orderBy("seq")
w_all = Window.partitionBy("protocolo")
ev_t = (ev_t.withColumn("data_anterior", F.lag("data_ts").over(w))
            .withColumn("seq_max", F.max("seq").over(w_all))
            .withColumn("n_aberturas", F.sum(F.when(F.col("acao") == "ABERTURA", 1).otherwise(0)).over(w_all)))

r = ev_t.agg(
    contar(F.col("seq").isNull()).alias("sequencia_nao_converte"),
    contar(F.col("data_ts").isNull()).alias("data_nao_converte"),
    contar(F.col("seq") != F.col("pos") + 1).alias("sequencia_fora_de_ordem"),
    contar(F.col("data_ts") < F.col("data_anterior")).alias("data_antes_da_anterior"),
    contar(F.col("data_ts") < F.col("abertura")).alias("evento_antes_abertura"),
    contar(F.col("data_ts") > F.col("fechamento")).alias("evento_apos_fechamento"),
    contar((F.col("seq") == F.col("seq_max")) & (F.col("data_ts") != F.col("fechamento"))).alias("ultimo_fora_do_fechamento"),
    contar(F.col("canal") != F.col("canal_at")).alias("canal_diferente_atendimento"),
    contar((F.col("seq") == 1) & (F.col("acao") != "ABERTURA")).alias("primeiro_nao_e_abertura"),
    contar((F.col("acao") == "ENCERRAMENTO") & (F.col("seq") != F.col("seq_max"))).alias("encerramento_nao_e_ultimo"),
    F.countDistinct(F.when(F.col("n_aberturas") > 1, F.col("protocolo"))).alias("protocolos_com_2_aberturas"),
).first()
print("\n== coerencia do historico")
for k, v in r.asDict().items():
    print(f"{k:<30}{v}")
print("eventos repetidos (protocolo, sequencia):", N_EV - ev.select("protocolo", "sequencia").distinct().count())

**O que descobrimos:**
- 636.857 eventos, entre 1 e 3 por atendimento, em quantidades quase iguais.
- Sequência, data, ação e canal convertem 100% e têm formato único. A data usa o formato com `T` (`aaaa-mm-ddThh:mm:ss`).
- A estrutura é coerente: sequência contínua começando em 1, nenhum evento repetido, datas em ordem e dentro do período do atendimento, último evento na data de fechamento e canal sempre igual ao do atendimento.
- A ordem das ações não faz sentido de negócio: em 254.625 atendimentos o primeiro evento não é ABERTURA, em 63.824 o ENCERRAMENTO não é o último e 15.309 têm mais de uma ABERTURA. As 5 ações aparecem em quantidades quase iguais, como se fossem distribuídas ao acaso.
- Na Silver, as ações ficam como estão e o problema é registrado no log. Nenhuma métrica deve ser calculada a partir da ação. Para tempo de atendimento, usar as datas de abertura e fechamento do atendimento.

## Raio-x das colunas de texto
**O que avaliamos:** rodamos o `perfilar_coluna` do `funcoes_comuns`, como pede o Padrão de Limpeza, nas colunas de texto das duas tabelas. Ele procura problemas que as outras checagens não pegam, como caracteres estranhos e encoding quebrado.

In [0]:
for df_, nome, cols in [
    (oc, "ocorrencias_rede", ["tipo", "causa", "alimentador"]),
    (at, "atendimentos", ["canal", "assunto", "subassunto"]),
]:
    for c in cols:
        print(f"\n######## {nome}.{c}")
        perfilar_coluna(df_, c, 10)

**O que descobrimos:**
- Nenhuma das seis colunas tem caractere estranho, encoding quebrado ou espaço nas pontas. Os valores de `tipo`, `causa`, `canal`, `assunto` e `subassunto` são exatamente os vistos nas seções de domínio.
- `alimentador` tem sempre 11 caracteres, no formato `ALM-<município>-999`.
- Três alimentadores concentram muito mais ocorrências que os outros: ALM-REC-008 (8.211), ALM-SAO-004 (4.664) e ALM-JAB-004 (3.913). O quarto colocado tem cerca de 1.470. Os três são dos municípios mais atingidos em julho de 2025, o que é investigado na seção de distribuição no tempo.

## Resumo da Exploração

### Ocorrências de rede (180.000 linhas)

| Problema | Linhas | Regra | Destino |
|---|---|---|---|
| Causa sem valor (nulo, `-`, `N/A`, `NULL`) | 5.400 | `valores_para_nulo` e depois NAO INFORMADA, diferente de NAO IDENTIFICADA | Corrigir |
| Duração com " min" escrito | 16.200 | Remover o " min" e `parse_inteiro` | Corrigir |
| Datas em texto | 180.000 | `parse_timestamp` | Corrigir |
| Interrupções acima de 24h | 274 | `flag_longa_duracao` | Flag |
| Interrupções de exatamente 48h, possível teto da origem | 30 | `flag_duracao_teto` | Flag |
| Interrupções sobrepostas na mesma UC | 329 | `flag_sobreposicao`, a Gold decide se junta os intervalos | Flag |
| Duração diferente das datas | 0 | Flag registrada no log | Flag |
| Encerramento antes da abertura | 0 | Flag registrada no log | Flag |
| Alimentador diferente do cadastro | 0 | Flag registrada no log | Flag |
| UC inexistente no cadastro | 0 | `separar_orfaos` | Quarentena |

### Atendimentos (318.871 linhas)

| Problema | Linhas | Regra | Destino |
|---|---|---|---|
| Datas em texto | 318.871 | `parse_timestamp` | Corrigir |
| Nota em texto | 309.305 | `parse_decimal` com uma casa | Corrigir |
| Nota nula | 9.566 | Fica nula, nunca zero | Manter |
| Fechamento antes da abertura | 0 | Flag registrada no log | Flag |
| UC inexistente no cadastro | 0 | `separar_orfaos` | Quarentena |

### Histórico dos atendimentos (636.857 eventos)

| Problema | Linhas | Regra | Destino |
|---|---|---|---|
| Eventos dentro de uma lista | 636.857 | Abrir em tabela filha `tb_atendimentos_historico` | Estrutura |
| Sequência e data em texto, data com `T` | 636.857 | `parse_inteiro` e `parse_timestamp` | Corrigir |
| Ordem das ações sem sentido de negócio | 254.625 atendimentos | Manter, registrar no log, não usar para métricas | Manter |
| Evento fora do período do atendimento | 0 | Flag registrada no log | Flag |
| Canal diferente do atendimento | 0 | Flag registrada no log | Flag |
| Protocolo inexistente nos atendimentos | 0 | `separar_orfaos` | Quarentena |

### Pontos para a Gold e o diagnóstico
- Julho de 2025 concentra 17.844 ocorrências (5 vezes o normal), 225 das 274 interrupções acima de 24h e 266 das 329 sobreposições. 72% delas estão em três alimentadores: ALM-REC-008, ALM-SAO-004 e ALM-JAB-004.
- Novembro de 2025 e fevereiro de 2026 têm cerca de 2.000 ocorrências a mais, crescendo igualmente em todos os tipos, causas e municípios, sem aumento nas reclamações. A troca de medidores foi testada e não explica o excesso. A origem segue em aberto.
- Os atendimentos de julho de 2025 triplicam em todos os assuntos, inclusive os sem relação com clima.
- 92% dos atendimentos de GERACAO DISTRIBUIDA vêm de UCs sem GD cadastrada. Em todos os assuntos, a parcela de UCs com GD é a mesma do cadastro (cerca de 7,8%), então o assunto do atendimento não tem relação com o perfil da UC.
